# Importación de librerías y carga de datos base

In [6]:
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

df_info = pd.read_csv('./../../dataset/oulad/studentInfo.csv')
df_vle = pd.read_csv('./../../dataset/oulad/studentVle.csv')

print("Datos cargados correctamente para CP-06.")

Datos cargados correctamente para CP-06.


# Definición de la variable objetivo y partición cronológica

In [3]:
df_info['target'] = df_info['final_result'].apply(lambda x: 1 if x == 'Withdrawn' else 0)

presentaciones_2013 = ['2013B', '2013J']
presentaciones_2014 = ['2014B', '2014J']

df_train_info = df_info[df_info['code_presentation'].isin(presentaciones_2013)].copy()
df_test_info = df_info[df_info['code_presentation'].isin(presentaciones_2014)].copy()

print(f"Estudiantes históricos (Train 2013): {len(df_train_info)}")
print(f"Estudiantes nuevos (Test 2014): {len(df_test_info)}")

Estudiantes históricos (Train 2013): 13529
Estudiantes nuevos (Test 2014): 19064


# Función de extracción de características temporales

In [4]:
def preparar_datos_temporales(df_vle, df_info_split, max_dias):
    vle_filtrado = df_vle[df_vle['date'] <= max_dias]

    vle_agg = vle_filtrado.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
    vle_agg.rename(columns={'sum_click': 'total_clicks'}, inplace=True)

    df_merged = pd.merge(df_info_split, vle_agg, on=['id_student', 'code_module', 'code_presentation'], how='left')
    df_merged['total_clicks'] = df_merged['total_clicks'].fillna(0)

    X = df_merged[['total_clicks']]
    y = df_merged['target']

    return X, y

print("Función de preparación de datos lista.")

Función de preparación de datos lista.


# Ejecución del experimento completo (MLP)

In [5]:
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import ADASYN

class ThresholdWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self

    def predict(self, X):
        proba = self.estimator.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

scoring_metrics = {
    'Accuracy': make_scorer(accuracy_score),
    'Precision': make_scorer(precision_score, zero_division=0),
    'Recall': make_scorer(recall_score, zero_division=0),
    'F1-Score': make_scorer(f1_score, zero_division=0)
}

param_grid_mlp = {
    'clasificador__estimator__hidden_layer_sizes': [(5,), (10,), (10, 5)],
    'clasificador__estimator__alpha': [0.0001, 0.01, 0.1],
    'clasificador__threshold': [0.3, 0.4, 0.5, 0.6]
}

ventanas_temporales = [30, 60, 90]
estrategias_balanceo = ['Línea Base (Sin balanceo)', 'Undersampling', 'ADASYN']

resultados_mejores = []
todas_las_permutaciones = []

print("Iniciando búsqueda exhaustiva CP-06 (MLP)...")

for dias in ventanas_temporales:
    print(f"--- Evaluando ventana a {dias} días ---")

    X_train, y_train = preparar_datos_temporales(df_vle, df_train_info, dias)
    X_test, y_test = preparar_datos_temporales(df_vle, df_test_info, dias)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for estrategia in estrategias_balanceo:
        if estrategia == 'Undersampling':
            sampler = RandomUnderSampler(random_state=42)
        elif estrategia == 'ADASYN':
            sampler = ADASYN(random_state=42, n_neighbors=5)
        else:
            sampler = None

        pasos = []
        if sampler is not None:
            pasos.append(('sampler', sampler))

        mlp_base = MLPClassifier(max_iter=1500, random_state=42)
        wrapper = ThresholdWrapper(estimator=mlp_base)
        pasos.append(('clasificador', wrapper))

        pipeline = ImbPipeline(pasos)

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid_mlp,
            cv=5,
            scoring=scoring_metrics,
            refit='Recall',
            n_jobs=-1,
            return_train_score=False
        )

        grid_search.fit(X_train_scaled, y_train)

        mejor_modelo = grid_search.best_estimator_
        y_pred = mejor_modelo.predict(X_test_scaled)

        resultados_mejores.append({
            'Días': dias,
            'Estrategia': estrategia,
            'Mejor Hidden_Layers': str(grid_search.best_params_['clasificador__estimator__hidden_layer_sizes']),
            'Mejor Alpha': grid_search.best_params_['clasificador__estimator__alpha'],
            'Mejor Threshold': grid_search.best_params_['clasificador__threshold'],
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1-Score': f1_score(y_test, y_pred, zero_division=0)
        })

        cv_results = grid_search.cv_results_
        for i in range(len(cv_results['params'])):
            todas_las_permutaciones.append({
                'Días': dias,
                'Estrategia': estrategia,
                'Hidden_Layers': str(cv_results['param_clasificador__estimator__hidden_layer_sizes'][i]),
                'Alpha': cv_results['param_clasificador__estimator__alpha'][i],
                'Threshold': cv_results['param_clasificador__threshold'][i],
                'CV_Accuracy': cv_results['mean_test_Accuracy'][i],
                'CV_Precision': cv_results['mean_test_Precision'][i],
                'CV_Recall': cv_results['mean_test_Recall'][i],
                'CV_F1': cv_results['mean_test_F1-Score'][i]
            })

        print(f"  > {estrategia}: Procesado.")

df_mejores = pd.DataFrame(resultados_mejores)
df_todas_permutaciones = pd.DataFrame(todas_las_permutaciones)

mejor_fila = df_mejores.sort_values(by=['Recall', 'F1-Score'], ascending=[False, False]).iloc[0]
mejor_dia = mejor_fila['Días']
mejor_estrategia = mejor_fila['Estrategia']
mejor_hidden = mejor_fila['Mejor Hidden_Layers']
mejor_alpha = mejor_fila['Mejor Alpha']

df_tradeoff = df_todas_permutaciones[
    (df_todas_permutaciones['Días'] == mejor_dia) &
    (df_todas_permutaciones['Estrategia'] == mejor_estrategia) &
    (df_todas_permutaciones['Hidden_Layers'] == mejor_hidden) &
    (df_todas_permutaciones['Alpha'] == mejor_alpha)
].copy()

df_tradeoff = df_tradeoff.sort_values(by=['Threshold'])

print("\n=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===")
display(df_mejores)

print(f"\n=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO ({mejor_dia} Días | {mejor_estrategia} | Layers={mejor_hidden} | Alpha={mejor_alpha}) ===")
display(df_tradeoff)

Iniciando búsqueda exhaustiva CP-06 (MLP)...
--- Evaluando ventana a 30 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.
--- Evaluando ventana a 60 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.
--- Evaluando ventana a 90 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.

=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===


,Días,Estrategia,Mejor Hidden_Layers,Mejor Alpha,Mejor Threshold,Accuracy,Precision,Recall,F1-Score
0,30,Línea Base (Sin balanceo),"(5,)",0.1000,0.3,0.679396,0.522098,0.599938,0.558318
1,30,Undersampling,"(10,)",0.0100,0.3,0.371433,0.348126,0.986799,0.514681
2,30,ADASYN,"(10,)",0.0001,0.3,0.351762,0.342034,0.995186,0.509097
3,60,Línea Base (Sin balanceo),"(10,)",0.1000,0.3,0.691828,0.535516,0.660351,0.591418
4,60,Undersampling,"(10,)",0.0100,0.3,0.411561,0.361550,0.969095,0.526627
5,60,ADASYN,"(10,)",0.0001,0.3,0.352549,0.342392,0.995962,0.509595
6,90,Línea Base (Sin balanceo),"(10,)",0.1000,0.3,0.710974,0.559804,0.675260,0.612136
7,90,Undersampling,"(10, 5)",0.0001,0.3,0.436425,0.371193,0.963348,0.535896
8,90,ADASYN,"(10,)",0.0001,0.3,0.400336,0.358371,0.981053,0.524973



=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO (60 Días | ADASYN | Layers=(10,) | Alpha=0.0001) ===


,Días,Estrategia,Hidden_Layers,Alpha,Threshold,CV_Accuracy,CV_Precision,CV_Recall,CV_F1
184,60,ADASYN,"(10,)",0.0001,0.3,0.326556,0.286514,0.979636,0.442409
185,60,ADASYN,"(10,)",0.0001,0.4,0.595751,0.431402,0.708730,0.503166
186,60,ADASYN,"(10,)",0.0001,0.5,0.667225,0.483302,0.629424,0.515927
187,60,ADASYN,"(10,)",0.0001,0.6,0.721035,0.547420,0.541609,0.516962
